In [9]:
# Reference: https://www.kaggle.com/snap/amazon-fine-food-reviews
# Reference: https://www.youtube.com/watch?v=QpzMWQvxXWk&t=204s&ab_channel=RobMulla
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')

import nltk 
from nltk import download
download('punkt')
download('averaged_perceptron_tagger')
download('maxent_ne_chunker')
download('words')

# Read in the data
df = pd.read_csv('../dataset/amazon_fine_food_reviews/amazon_reviews.csv')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/thomashazekamp/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/thomashazekamp/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /Users/thomashazekamp/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     /Users/thomashazekamp/nltk_data...
[nltk_data]   Package words is already up-to-date!


In [10]:
example = df['Text'][50]

df = df.head(500) # Only top 500 reviews - for testing purposes

tokens = nltk.word_tokenize(example)

nltk_pos_tagged = nltk.pos_tag(tokens) # Part of speech tagging - gives each token a tag (e.g. noun, verb, etc.)

entities = nltk.chunk.ne_chunk(nltk_pos_tagged) # Named entity recognition - identifies named entities (e.g. person, place, etc.)

In [11]:
## Using Vader to perform sentiment analysis
# - Vader does not account the relationship between words in a sentence, so it is not as accurate as other methods

from nltk.sentiment import SentimentIntensityAnalyzer
from tqdm.notebook import tqdm

download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

sia.polarity_scores(example) # Using Vader to perform sentiment analysis on the example review, showing negative, neutral, positive, and compound scores (-1 to 1 on how negative/positive the review is)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/thomashazekamp/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


{'neg': 0.22, 'neu': 0.78, 'pos': 0.0, 'compound': -0.5448}

In [12]:
# Using the Roberta model from huggingface

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax


In [13]:
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

In [14]:
# Run Roberta model

encoded_text = tokenizer(example, return_tensors='pt')
output = model(**encoded_text)
scores = output[0][0].detach().numpy()
scores = softmax(scores)
scores_dict = {
    "roberta_negative": scores[0],
    "roberta_neutral": scores[1],
    "roberta_positive": scores[2]
}
print(scores_dict)

{'roberta_negative': 0.97635514, 'roberta_neutral': 0.020687457, 'roberta_positive': 0.0029573694}


In [15]:
def polarity_scores_roberta(example):
    encoded_text = tokenizer(example, return_tensors='pt')
    output = model(**encoded_text)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    scores_dict = {
        "roberta_negative": scores[0],
        "roberta_neutral": scores[1],
        "roberta_positive": scores[2]
    }
    return scores_dict

In [21]:
res = {}
for i, row in tqdm(df.iterrows()):
    try:
        text = row['Text']
        myid = row['Id']
        vader_result = sia.polarity_scores(text)
        vader_result_rename = {}
        for key, value in vader_result.items():
            vader_result_rename[f"vader_{key}"] = value

        roberta_result = polarity_scores_roberta(text)
        both = {**vader_result_rename, **roberta_result}
        
        res[myid] = both # Add the results to the dictionary holding both the vader and roberta results
    except RuntimeError:
        print(f'Broke on id {myid}')

res # Show the dictionary of the polarity scores for each review

0it [00:00, ?it/s]

Broke on id 83
Broke on id 187


{1: {'vader_neg': 0.0,
  'vader_neu': 0.695,
  'vader_pos': 0.305,
  'vader_compound': 0.9441,
  'roberta_negative': 0.009624226,
  'roberta_neutral': 0.049980376,
  'roberta_positive': 0.94039536},
 2: {'vader_neg': 0.138,
  'vader_neu': 0.862,
  'vader_pos': 0.0,
  'vader_compound': -0.5664,
  'roberta_negative': 0.5089859,
  'roberta_neutral': 0.4524137,
  'roberta_positive': 0.038600426},
 3: {'vader_neg': 0.091,
  'vader_neu': 0.754,
  'vader_pos': 0.155,
  'vader_compound': 0.8265,
  'roberta_negative': 0.0032289105,
  'roberta_neutral': 0.09806752,
  'roberta_positive': 0.8987036},
 4: {'vader_neg': 0.0,
  'vader_neu': 1.0,
  'vader_pos': 0.0,
  'vader_compound': 0.0,
  'roberta_negative': 0.0022951283,
  'roberta_neutral': 0.090219304,
  'roberta_positive': 0.9074856},
 5: {'vader_neg': 0.0,
  'vader_neu': 0.552,
  'vader_pos': 0.448,
  'vader_compound': 0.9468,
  'roberta_negative': 0.0016347276,
  'roberta_neutral': 0.010302456,
  'roberta_positive': 0.98806286},
 6: {'vader_

In [18]:
# Add the results of both the vader and roberta to the dataframe

results_df = pd.DataFrame(res).T # Convert the dictionary to a dataframe
results_df = results_df.reset_index().rename(columns={'index':'Id'}) # Reset the index and rename the column to 'Id'
results_df = results_df.merge(df, how='left') # Merge the dataframe with the original dataframe

{'vader_neg': 0.0,
 'vader_neu': 0.695,
 'vader_pos': 0.305,
 'vader_compound': 0.9441,
 'roberta_negative': 0.009624226,
 'roberta_neutral': 0.049980376,
 'roberta_positive': 0.94039536}